# Preproccessing of the dataset

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import pandas as pd
import numpy as np

from egg_hunt.config import IMG_SIZE, NUM_CLASSES, RAW_DATA, CACHE
from egg_hunt.preprocessing import (load_split, load_image, crop_egg,
                                    preprocess_image, get_cropped_arrays)

# ---------------------------------------
# Loading the image data
# ---------------------------------------

In [ ]:
train_data_path   = RAW_DATA / 'train/train_data'
train_labels_path = RAW_DATA / 'train/train_labels.json'
test_data_path    = RAW_DATA / 'test/test_data'
test_labels_path  = RAW_DATA / 'test/test_labels.json'

In [ ]:
train_df = load_split(train_labels_path, train_data_path)
test_df  = load_split(test_labels_path,  test_data_path)

In [ ]:
train_df.head(3)

# ---------------------------------------
# Visualisation
# ---------------------------------------

In [ ]:
def show_with_box(row: pd.Series):
    img = load_image(row['path'])
    x, y, w, h = row['bbox']

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='blue', facecolor='none'))
    ax.axis('off')
    plt.show()

show_with_box(train_df.iloc[334])

In [ ]:
# the cropped egg, i.e. what a user would upload
crop = crop_egg(train_df.iloc[334])
crop

In [ ]:
# what the model sees (square, 224x224)
Image.fromarray(preprocess_image(crop))

### This code is how we get all image_ids that contain 2 or more eggs in one image.

  **`multiple_eggs = train_df.value_counts('image_id').sort_values(ascending=False).head(30)`**

# ---------------------------------------
# One-Hot Encoding, Arrays
# ---------------------------------------

In [ ]:
X_train, y_train, X_test, y_test = get_cropped_arrays(train_df, test_df, cache_path=CACHE)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
# Compare: centred crop (no jitter) vs. the same egg from the jittered X_train cache

N = 6                                    # how many eggs to show
rng_pick = np.random.default_rng(0)      # change the seed to see other eggs
idx = rng_pick.choice(len(train_df), size=N, replace=False)

fig, axes = plt.subplots(2, N, figsize=(3 * N, 6.5))
n_identical = 0

for col, i in enumerate(idx):
    row = train_df.iloc[i]                               # X_train[i] belongs to train_df.iloc[i]
    centred  = preprocess_image(crop_egg(row))           # jitter=0 -> normal centred crop
    jittered = X_train[i]                                # same egg, from get_cropped_arrays()
    n_identical += np.array_equal(centred, jittered)

    for r, (img, label) in enumerate([(centred, "centred"), (jittered, "jittered")]):
        ax = axes[r, col]
        ax.imshow(img)
        ax.axvline(IMG_SIZE / 2, color="red", lw=0.6)    # crosshair = image centre,
        ax.axhline(IMG_SIZE / 2, color="red", lw=0.6)    # makes the shift easy to see
        ax.set_title(f"#{i} {label}", fontsize=9)
        ax.axis("off")
    axes[0, col].set_xlabel(row["category_name"])

plt.suptitle("Top: centred crop  |  Bottom: jittered crop from X_train", y=1.0)
plt.tight_layout()
plt.show()

if n_identical == N:
    print("⚠️ All pairs are identical – the cache contains no jitter. "
          "Delete the cache file and run get_cropped_arrays() again.")